In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

In [3]:
df = pd.read_csv("../../datasets/npk/NPK_dataset.csv")
print("Dataset Loaded Successfully")
print(df.head())

Dataset Loaded Successfully
   Nitrogen  Phosphorus  Potassium  Temperature   Humidity    Rainfall  \
0     66.80       93.95     150.09    17.261834  72.941652  302.842639   
1     72.04       67.86      83.08    21.846116  99.361954   94.693847   
2     92.02       66.56      88.17    33.246895  81.506836   83.563685   
3     67.76       45.32      56.07    14.396289  59.274465   31.508836   
4     31.86       62.98     150.65    16.773218  51.191584  295.193482   

        Crop Soil_Type    Variety  Soil_Moisture  
0      Wheat      Clay   Soft Red          78.31  
1     Tomato      Clay  Beefsteak          64.07  
2  Sugarcane      Clay   Co 86032          78.33  
3  Sugarcane      Silt    Co 0238          44.63  
4      Maize     Sandy      Sweet          33.28  


In [4]:
input_features = [
    "Temperature",
    "Humidity",
    "Rainfall",
    "Crop",
    "Soil_Type",
    "Variety",
    "Soil_Moisture"
]


In [5]:
target_features = [
    "Nitrogen",
    "Phosphorus",
    "Potassium"
]

In [6]:
X = df[input_features]
y = df[target_features]

In [7]:
categorical_columns = [
    "Crop",
    "Soil_Type",
    "Variety"
]

numerical_columns = [
    "Temperature",
    "Humidity",
    "Rainfall",
    "Soil_Moisture"
]

In [8]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_columns
        ),
        (
            "num",
            "passthrough",
            numerical_columns
        )
    ]
)

In [29]:
rf_model = RandomForestRegressor(
    n_estimators=150,
    max_depth=12,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

In [30]:
model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", MultiOutputRegressor(rf_model))
])


In [31]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [32]:
print("\nTraining Model...\n")

model.fit(X_train, y_train)

print("Model Training Completed")


Training Model...

Model Training Completed


In [33]:
y_pred = model.predict(X_test)

In [34]:
train_pred = model.predict(X_train)

train_r2 = r2_score(y_train, train_pred)
train_accuracy = train_r2 * 100

train_mae = mean_absolute_error(y_train, train_pred)

print("\n===================================")
print("TRAINING PERFORMANCE")
print("===================================")

print(f"Training MAE      : {train_mae:.2f}")
print(f"Training R2 Score : {train_r2:.4f}")
print(f"Training Accuracy : {train_accuracy:.2f}%")


TRAINING PERFORMANCE
Training MAE      : 3.82
Training R2 Score : 0.9623
Training Accuracy : 96.23%


In [35]:
test_pred = model.predict(X_test)

test_r2 = r2_score(y_test, test_pred)
test_accuracy = test_r2 * 100

test_mae = mean_absolute_error(y_test, test_pred)

print("\n===================================")
print("TEST PERFORMANCE")
print("===================================")

print(f"Test MAE      : {test_mae:.2f}")
print(f"Test R2 Score : {test_r2:.4f}")
print(f"Test Accuracy : {test_accuracy:.2f}%")


TEST PERFORMANCE
Test MAE      : 4.40
Test R2 Score : 0.9487
Test Accuracy : 94.87%


In [36]:
print("\n========== MODEL EVALUATION ==========\n")

for i, target in enumerate(target_features):

    mae = mean_absolute_error(y_test.iloc[:, i], test_pred[:, i])

    r2 = r2_score(y_test.iloc[:, i], test_pred[:, i])

    accuracy = r2 * 100

    print(f"\n----- {target} -----")

    print(f"MAE          : {mae:.2f}")
    print(f"R2 Score     : {r2:.4f}")
    print(f"Accuracy     : {accuracy:.2f}%")


========== MODEL EVALUATION ==========


----- Nitrogen -----
MAE          : 3.26
R2 Score     : 0.9294
Accuracy     : 92.94%

----- Phosphorus -----
MAE          : 3.39
R2 Score     : 0.9562
Accuracy     : 95.62%

----- Potassium -----
MAE          : 6.55
R2 Score     : 0.9606
Accuracy     : 96.06%


In [37]:
joblib.dump(
    model,
    "npk_random_forest_model.pkl",
    compress=3
)

print("\nModel Saved Successfully")
print("File Name : npk_random_forest_model.pkl")


Model Saved Successfully
File Name : npk_random_forest_model.pkl


In [38]:
sample_data = pd.DataFrame({
    "Temperature": [40],
    "Humidity": [50],
    "Rainfall": [340],
    "Crop": ["Rice"],
    "Soil_Type": ["Clay"],
    "Variety": ["Basmati"],
    "Soil_Moisture": [79]
})

prediction = model.predict(sample_data)

print("\n========== SAMPLE PREDICTION ==========\n")

print(f"Nitrogen   : {prediction[0][0]:.2f}")
print(f"Phosphorus : {prediction[0][1]:.2f}")
print(f"Potassium  : {prediction[0][2]:.2f}")


========== SAMPLE PREDICTION ==========

Nitrogen   : 76.47
Phosphorus : 95.22
Potassium  : 161.28
